<a href="https://www.kaggle.com/code/avikdas567/cafa-6-protein-function-prediction?scriptVersionId=293786720" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ============================================================
# CAFA 6 Protein Function Prediction
# ============================================================

import pandas as pd
import numpy as np
from tqdm import tqdm

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
DATA_ROOT = "/kaggle/input/cafa-6-protein-function-prediction"
TRAIN_TERMS = f"{DATA_ROOT}/Train/train_terms.tsv"
TEST_FASTA  = f"{DATA_ROOT}/Test/testsuperset.fasta"
OUT_FILE = "submission.tsv"

# ------------------------------------------------------------
# Load training annotations
# ------------------------------------------------------------
print("Loading training annotations...")
train_terms = pd.read_csv(TRAIN_TERMS, sep="\t")
train_terms.columns = ["protein_id", "go_id", "aspect"]

# ------------------------------------------------------------
# Compute GO term frequencies
# ------------------------------------------------------------
term_stats = (
    train_terms
    .groupby(["aspect", "go_id"])
    .size()
    .reset_index(name="count")
)

term_stats["prob"] = (
    term_stats.groupby("aspect")["count"]
    .transform(lambda x: x / x.max())
)

# ------------------------------------------------------------
# Aggressive cap
# ------------------------------------------------------------
TOP_K = {
    "F": 50,   # Molecular Function
    "P": 100,  # Biological Process
    "C": 50    # Cellular Component
}

top_terms = []
for aspect, k in TOP_K.items():
    df = (
        term_stats[term_stats["aspect"] == aspect]
        .sort_values("prob", ascending=False)
        .head(k)
    )
    top_terms.append(df[["go_id", "prob"]])

top_terms = pd.concat(top_terms, ignore_index=True)

print("Total GO terms used:", len(top_terms))

# ------------------------------------------------------------
# FAST FASTA ID parsing
# ------------------------------------------------------------
def read_fasta_ids(path):
    with open(path) as f:
        return [line[1:].split()[0] for line in f if line.startswith(">")]

print("Reading test FASTA...")
test_ids = read_fasta_ids(TEST_FASTA)
print("Test proteins:", len(test_ids))

# ------------------------------------------------------------
# Vectorized cross join
# ------------------------------------------------------------
print("Building submission (vectorized)...")

test_df = pd.DataFrame({"protein_id": test_ids})
test_df["key"] = 1
top_terms["key"] = 1

submission = (
    test_df
    .merge(top_terms, on="key")
    .drop(columns="key")
)

submission["prob"] = submission["prob"].round(3)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
submission.to_csv(
    OUT_FILE,
    sep="\t",
    header=False,
    index=False
)

print("\n Submission written:", OUT_FILE)
print("Total rows:", len(submission))
submission.head(10)

Loading training annotations...
Total GO terms used: 200
Reading test FASTA...
Test proteins: 224309
Building submission (vectorized)...

 Submission written: submission.tsv
Total rows: 44861800


,protein_id,go_id,prob
0,A0A0C5B5G6,GO:0005515,1.000
1,A0A0C5B5G6,GO:0042802,0.105
2,A0A0C5B5G6,GO:0042803,0.048
3,A0A0C5B5G6,GO:0003723,0.048
4,A0A0C5B5G6,GO:0003677,0.044
5,A0A0C5B5G6,GO:0003700,0.036
6,A0A0C5B5G6,GO:0043565,0.034
7,A0A0C5B5G6,GO:0003729,0.034
8,A0A0C5B5G6,GO:0000976,0.032
9,A0A0C5B5G6,GO:0000978,0.026
